# Auxiliary-table ablation

Run Jupyter from the repository root. This is a coarse walkthrough, not a tuning study. TabM is omitted.

Pipeline: `load_data -> make_listing -> load_auxiliary -> make_auxiliary -> attach_auxiliary -> [scale_numerical] -> [onehot_encoding] -> encode_high_card`.

Encoding is fixed so that only the attached tables change:

- All models: low-card one-hot, high-card ordinal.
- Linear / Ridge additionally scale numeric columns; tree / boosting models do not.

Table-level groups:

1. listing only
2. listing + all joined
3. listing + all joined − one auxiliary table (HDB, MRT, schools, malls, COE, stock). MRT / schools / malls need HDB coordinates, so **− HDB** is listing + COE + stock.

In [1]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

from src.auxiliary import attach_auxiliary, load_auxiliary
from src.evaluation import evaluate
from src.features import (
    encode_high_card,
    feature_groups,
    make_auxiliary,
    make_listing,
    onehot_encoding,
    order_features,
    scale_numerical,
)
from src.validation import load_data

## Load and build feature tables

`make_auxiliary` turns each cleaned auxiliary table into a join-ready table: COE becomes one monthly `coe_sum`, stock prices one monthly equal-weight `stock_index`, MRT is collapsed to one row per station with `LINE_COUNT`, schools are flagged primary / secondary. `attach_auxiliary` joins only the tables in the dict it is given.

In [2]:
train_df, valid_df = load_data()
y_train = train_df['MONTHLY_RENT']
y_valid = valid_df['MONTHLY_RENT']

X_train = make_listing(train_df)
X_valid = make_listing(valid_df)

raw = load_auxiliary()
auxiliary = make_auxiliary(
    df_coe=raw['coe'],
    df_mrt=raw['mrt'],
    df_schools=raw['schools'],
    df_hdb=raw['hdb'],
    df_malls=raw['malls'],
    df_stock=raw['stock'],
)

TABLE_SETS = {'listing only': ()}
TABLE_SETS['all joined'] = tuple(auxiliary)
TABLE_SETS['all - hdb'] = ('coe', 'stock')
for name in ('mrt', 'schools', 'malls', 'coe', 'stock'):
    TABLE_SETS[f'all - {name}'] = tuple(table for table in auxiliary if table != name)


def build(tables):
    selected = {name: auxiliary[name] for name in tables}
    return tuple(order_features(attach_auxiliary(X, selected)) for X in (X_train, X_valid))


feature_sets = {name: build(tables) for name, tables in TABLE_SETS.items()}
for name, (X_fit, X_eval) in feature_sets.items():
    assert list(X_fit.columns) == list(X_eval.columns)
    assert X_fit.index.equals(y_train.index) and X_eval.index.equals(y_valid.index)
    print(f'{name:13s} {X_fit.shape[1]:2d} columns', TABLE_SETS[name])

numeric, low_card, high_card = feature_groups(feature_sets['all joined'][0])
print('numeric  ', numeric)
print('low_card ', low_card)
print('high_card', high_card)

listing only   8 columns ()
all joined    24 columns ('coe', 'stock', 'mrt', 'schools', 'hdb', 'malls')
all - hdb     10 columns ('coe', 'stock')
all - mrt     21 columns ('coe', 'stock', 'schools', 'hdb', 'malls')
all - schools 20 columns ('coe', 'stock', 'mrt', 'hdb', 'malls')
all - malls   22 columns ('coe', 'stock', 'mrt', 'schools', 'hdb')
all - coe     23 columns ('stock', 'mrt', 'schools', 'hdb', 'malls')
all - stock   23 columns ('coe', 'mrt', 'schools', 'hdb', 'malls')
numeric   ['FLOOR_AREA_SQM', 'flat_age', 'month_index', 'LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'mrt_distance_km', 'mrt_line_count', 'primary_school_distance_km', 'secondary_school_distance_km', 'schools_within_1km', 'schools_within_2km', 'mall_distance_km', 'coe_sum', 'stock_index']
low_card  ['TOWN', 'FLAT_TYPE', 'FLAT_MODEL', 'PLANNING_AREA']
high_card ['BLOCK', 'STREET', 'SUBZONE', 'mrt_name', 'mall_name']


## Models

In [3]:
LINEAR_MODELS = ('Linear Regression', 'Ridge Regression')

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1,
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=100, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1,
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.08, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=42,
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        random_state=42, n_jobs=-1, verbosity=-1,
    ),
    'CatBoost': CatBoostRegressor(
        iterations=300, learning_rate=0.05, depth=8,
        loss_function='RMSE', random_seed=42, thread_count=-1, verbose=False,
    ),
}


def encode(X_fit, X_eval, linear):
    if linear:
        X_fit, X_eval = scale_numerical(X_fit, X_eval)
    X_fit, X_eval = onehot_encoding(X_fit, X_eval)
    return encode_high_card(X_fit, X_eval, method='ordinal')


def fit_models(X_train, X_valid, *, names=None):
    selected = models if names is None else {name: models[name] for name in names}
    rows = []
    for name, model in selected.items():
        start = time.perf_counter()
        X_fit, X_eval = encode(X_train, X_valid, linear=name in LINEAR_MODELS)
        estimator = clone(model).fit(X_fit, y_train)
        predictions = estimator.predict(X_eval)
        elapsed = time.perf_counter() - start

        metrics = evaluate(y_valid, predictions)
        assert np.isfinite(list(metrics.values())).all()
        rows.append({'Model': name, **metrics, 'Training Time (s)': elapsed})
    return pd.DataFrame(rows)

## Table-level ablation

In [4]:
results = []
for set_name, (X_fit, X_eval) in feature_sets.items():
    table = fit_models(X_fit, X_eval)
    table.insert(0, 'Tables', set_name)
    results.append(table)
    print(set_name, table.set_index('Model')['RMSE'].round(2).to_dict())
results = pd.concat(results, ignore_index=True)

/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


...

/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/dev/miniconda3/envs/kaggle/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


all - stock {'Linear Regression': 530.53, 'Ridge Regression': 530.56, 'Random Forest': 481.99, 'Extra Trees': 482.1, 'HistGradientBoosting': 478.24, 'LightGBM': 476.91, 'CatBoost': 477.94}


RMSE by model (rows) and table set (columns); lower is better.

In [5]:
rmse = results.pivot(index='Model', columns='Tables', values='RMSE')
rmse = rmse.loc[list(models), list(TABLE_SETS)]
rmse.round(2)

Tables,listing only,all joined,all - hdb,all - mrt,all - schools,all - malls,all - coe,all - stock
Model,,,,,,,,
Linear Regression,615.65,518.78,524.07,520.60,518.80,519.53,608.79,530.53
Ridge Regression,615.73,518.81,524.12,520.64,518.83,519.56,608.85,530.56
Random Forest,490.11,480.23,488.80,480.78,479.65,480.87,481.12,481.99
Extra Trees,493.76,480.04,492.59,480.95,480.49,479.81,480.80,482.10
HistGradientBoosting,481.90,476.11,481.78,476.72,476.73,475.90,476.78,478.24
LightGBM,481.17,476.15,481.45,476.79,476.46,475.93,475.23,476.91
CatBoost,483.03,476.72,482.75,478.21,476.34,477.16,475.99,477.94


RMSE change versus **all joined** (positive = removing that table hurts).

In [6]:
(rmse.sub(rmse['all joined'], axis=0)).drop(columns='all joined').round(2)

Tables,listing only,all - hdb,all - mrt,all - schools,all - malls,all - coe,all - stock
Model,,,,,,,
Linear Regression,96.87,5.29,1.82,0.02,0.75,90.01,11.75
Ridge Regression,96.92,5.31,1.83,0.02,0.75,90.04,11.75
Random Forest,9.88,8.56,0.55,-0.58,0.64,0.89,1.76
Extra Trees,13.72,12.55,0.91,0.45,-0.23,0.76,2.06
HistGradientBoosting,5.79,5.67,0.61,0.62,-0.21,0.67,2.13
LightGBM,5.02,5.30,0.64,0.31,-0.22,-0.92,0.76
CatBoost,6.31,6.03,1.49,-0.38,0.44,-0.73,1.22
